# MCMC BAO Only (Paper I Dataset Separation)**Author**: Ricardo Alvim**Purpose**: Constrain Evaporating Universe using BAO data alone---## Runtime: ~2-3 hours on Colab Pro

In [ ]:
# ============================================================
# INSTALLATION (run this cell first!)
# ============================================================
# Core packages are pre-installed in Colab
# !pip install numpy matplotlib scipy  # Already in Colab
print('Colab environment ready!')

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.integrate import quadfrom scipy.optimize import minimizeimport emceefrom multiprocessing import Pool, cpu_countimport cornerimport jsonfrom datetime import datetimeimport warningswarnings.filterwarnings('ignore')print("="*70)print("MCMC BAO ONLY - Evaporating Universe")print("="*70)n_cores = min(cpu_count(), 32)print(f'Available CPU cores: {n_cores}')

In [ ]:
# =============================================================# BAO DATA# =============================================================# BAO measurements (z, DV/rd, error)# From SDSS, 6dFGS, WiggleZ, BOSS DR12bao_data = np.array([[0.106, 2.98, 0.13],   # 6dFGS[0.15, 4.47, 0.17],    # SDSS MGS[0.38, 10.23, 0.17],   # BOSS DR12[0.51, 13.36, 0.21],   # BOSS DR12[0.61, 15.45, 0.24],   # BOSS DR12[2.33, 37.6, 1.3],     # Ly-alpha])z_bao = bao_data[:, 0]DV_rd_obs = bao_data[:, 1]DV_rd_err = bao_data[:, 2]print(f"BAO data points: {len(z_bao)}")

In [ ]:
# =============================================================# EVAPORATING UNIVERSE MODEL# =============================================================c = 299792.458  # km/sdef w_de(z, w0, z_trans):if z_trans <= 0.01:return -1.0if z > z_trans:return -1.0delta_w = w0 - (-1.0)return -1.0 + delta_w * (1 - z/z_trans)**2def E_z(z, Omega_m, w0, z_trans):Omega_de = 1 - Omega_mw = w_de(z, w0, z_trans)rho_de = Omega_de * (1 + z)**(3*(1+w))return np.sqrt(Omega_m * (1+z)**3 + rho_de)def comoving_distance(z, H0, Omega_m, w0, z_trans):def integrand(zp):return 1.0 / E_z(zp, Omega_m, w0, z_trans)result, _ = quad(integrand, 0, z, limit=200)return c / H0 * resultdef DV(z, H0, Omega_m, w0, z_trans):"""Volume-averaged distance."""DM = comoving_distance(z, H0, Omega_m, w0, z_trans)Ez = E_z(z, Omega_m, w0, z_trans)DH = c / (H0 * Ez)return (z * DM**2 * DH)**(1/3)print("Model defined.")

In [ ]:
# =============================================================# LIKELIHOOD# =============================================================rd_fid = 147.0  # Mpc (fiducial sound horizon)def log_likelihood(theta):H0, Omega_m, w0, z_trans, rd = thetachi2 = 0for i, z in enumerate(z_bao):DV_pred = DV(z, H0, Omega_m, w0, z_trans)DV_rd_pred = DV_pred / rdchi2 += ((DV_rd_pred - DV_rd_obs[i]) / DV_rd_err[i])**2return -0.5 * chi2def log_prior(theta):H0, Omega_m, w0, z_trans, rd = thetaif not (60 < H0 < 80): return -np.infif not (0.2 < Omega_m < 0.4): return -np.infif not (-1.5 < w0 < -1.0): return -np.infif not (0.1 < z_trans < 0.5): return -np.infif not (140 < rd < 155): return -np.infreturn 0.0def log_probability(theta):lp = log_prior(theta)if not np.isfinite(lp):return -np.infreturn lp + log_likelihood(theta)print("Likelihood defined.")

In [ ]:
#=============================================================#RUNMCMC#=============================================================#Initialguessinitial=np.array([73.0,0.30,-1.15,0.22,147.0])ndim=len(initial)nwalkers=32nsteps=5000#Initializewalkerspos=initial+1e-3*np.random.randn(nwalkers,ndim)print(f"RunningMCMC:{nwalkers}walkers,{nsteps}steps...")print("Thiswilltake~2-3hoursonColabPro")withPool(n_cores)aspool:sampler=emcee.EnsembleSampler(nwalkers,ndim,log_probability,pool=pool)sampler.run_mcmc(pos,nsteps,progress=True)print("MCMCcomplete!")

In [ ]:
# =============================================================# ANALYZE RESULTS# =============================================================# Discard burn-inburnin = 1000samples = sampler.get_chain(discard=burnin, flat=True)labels = [r'$H_0$', r'$\Omega_m$', r'$w_0$', r'$z_{trans}$', r'$r_d$']# Best fitmeans = np.mean(samples, axis=0)stds = np.std(samples, axis=0)print("\n" + "="*50)print("BAO ONLY RESULTS")print("="*50)for i, (label, mean, std) in enumerate(zip(labels, means, stds)):print(f"{label}: {mean:.4f} +/- {std:.4f}")

In [ ]:
# =============================================================# CORNER PLOT# =============================================================fig = corner.corner(samples, labels=labels, quantiles=[0.16, 0.5, 0.84],show_titles=True, title_fmt='.3f')plt.suptitle('BAO Only Constraints', fontsize=14)plt.tight_layout()plt.savefig('mcmc_bao_only_corner.png', dpi=150)plt.show()

In [ ]:
# =============================================================# SAVE RESULTS# =============================================================results = {"metadata": {"analysis": "MCMC BAO Only","date": datetime.now().isoformat(),"nwalkers": nwalkers,"nsteps": nsteps,"burnin": burnin},"parameters": {"H0": [float(means[0]), float(stds[0])],"Omega_m": [float(means[1]), float(stds[1])],"w0": [float(means[2]), float(stds[2])],"z_trans": [float(means[3]), float(stds[3])],"rd": [float(means[4]), float(stds[4])]},"maturity": "Paper Standard","figures": ["mcmc_bao_only_corner.png"]}with open('mcmc_bao_only_results.json', 'w') as f:json.dump(results, f, indent=2)# Save chain for laternp.save('mcmc_bao_only_chain.npy', samples)print("Saved results!")try:from google.colab import filesfiles.download('mcmc_bao_only_corner.png')files.download('mcmc_bao_only_results.json')files.download('mcmc_bao_only_chain.npy')except:print("Files saved locally.")